# Q2 Numerical Solution and Validation

Confirmed production route:

**cylindrical cell-centered FVM**
→ **joint state \(y=[T,C]\)**
→ **local coupled surface Picard for \((T_s,C_s)\)**
→ **Method of Lines**
→ **SciPy BDF**

Independent auxiliary analysis:

**Backward Euler + block Picard**

This notebook is organized as a numerical acceptance workflow. The Q2 model is solved independently from \(t=0\) with Appendix 3 properties over the full 0–3 h interval.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path.cwd()
if not (PROJECT / "src").exists() and (PROJECT.parent / "src").exists():
    PROJECT = PROJECT.parent
if not (PROJECT / "src").exists():
    raise RuntimeError("Run this notebook from the q2_drying_project root or notebooks/ directory.")

sys.path.insert(0, str(PROJECT / "src"))

from q2_solver import (
    EnvironmentData, Q2Parameters, SolverConfig, PicardConfig,
    density, heat_capacity, thermal_conductivity, moisture_diffusivity,
    solve_q2, solve_q2_block_picard,
    paper_tables, physical_diagnostics, property_ranges,
    boundary_residuals, balance_diagnostics,
    grid_convergence_study, compare_solutions,
    face_average_sensitivity, picard_error_study, splitting_error_study,
    augmented_snapshot, compare_bdf_picard,
    write_result2, explicit_stability_reference,
)

ENV_PATH = PROJECT / "data" / "附件1.xlsx"
OUT_DIR = PROJECT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)
print("Environment file exists:", ENV_PATH.exists())


## 1. Check Attachment 1 for Q2

Q2 needs 0–10800 s. The model uses piecewise-linear interpolation of the measured ambient temperature and moisture datum. No extrapolation should be required.


In [ ]:
env = EnvironmentData.from_excel(ENV_PATH)
mask = env.t <= 10800
print("Rows in file:", len(env.t))
print("Full time range:", (env.t[0], env.t[-1]))
print("Unique time increments:", np.unique(np.diff(env.t)))
print("Q2 sample count:", int(mask.sum()))
print("Q2 ambient T:", env.T_inf(0), "->", env.T_inf(10800))
print("Q2 ambient C datum:", env.C_inf(0), "->", env.C_inf(10800))


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(env.t[mask] / 3600, env.temperature[mask], marker="o", markersize=2)
plt.xlabel("Time (h)")
plt.ylabel("Ambient temperature (degC)")
plt.title("Q2 ambient temperature")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(env.t[mask] / 3600, env.moisture[mask], marker="o", markersize=2)
plt.xlabel("Time (h)")
plt.ylabel("Ambient moisture datum (kg/kg)")
plt.title("Q2 ambient moisture datum")
plt.grid(True, alpha=0.3)
plt.show()


## 2. Appendix 3 material properties

The solver stores temperature in degC, but the diffusivity function converts internally to Kelvin:

\[
D(C,T)=2.4\times10^{-3}\exp(-0.45/C)\exp[-3850/(T+273.15)].
\]

Check the property magnitudes before solving the PDE.


In [ ]:
p = Q2Parameters()
C_grid = np.linspace(0.15, 2.55, 300)
print("Initial rho:", density(p.C0))
print("Initial cp:", heat_capacity(p.C0))
print("Initial k:", thermal_conductivity(p.C0))
print("Initial D:", moisture_diffusivity(p.C0, p.T0_C))
print("Initial alpha:", thermal_conductivity(p.C0) / (density(p.C0) * heat_capacity(p.C0)))

plt.figure(figsize=(7, 4))
plt.plot(C_grid, density(C_grid), label="rho")
plt.xlabel("Moisture content C (kg/kg)")
plt.ylabel("Density (kg/m^3)")
plt.title("Density relation")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(C_grid, heat_capacity(C_grid), label="cp")
plt.xlabel("Moisture content C (kg/kg)")
plt.ylabel("Heat capacity (J/(kg K))")
plt.title("Heat-capacity relation")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(C_grid, thermal_conductivity(C_grid), label="k")
plt.xlabel("Moisture content C (kg/kg)")
plt.ylabel("Thermal conductivity (W/(m K))")
plt.title("Thermal-conductivity relation")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
for T0 in [30, 40, 50]:
    plt.semilogy(C_grid, moisture_diffusivity(C_grid, T0), label=f"{T0} degC")
plt.xlabel("Moisture content C (kg/kg)")
plt.ylabel("Diffusivity D (m^2/s)")
plt.title("Moisture diffusivity from Appendix 3")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 3. Numerical scales

The requested 0.1 cm is an output spacing, not the PDE mesh width. The explicit estimates below are only reference values to show why the production method uses an implicit stiff integrator.


In [ ]:
for N in [160, 320, 640, 1280]:
    print(explicit_stability_reference(N, p))


## 4. Production BDF baseline

Start with \(N=320\) for a quick acceptance run. After the diagnostics pass, run \(N=640\) and \(N=1280\) for convergence and final production.

The default BDF workflow restarts at each 60 s ambient-data knot. Internally it remains adaptive; `max_step=5 s` is an upper bound, not the output interval.


In [ ]:
cfg = SolverConfig(
    method="BDF",
    rtol=1e-9,
    atol=1e-11,
    max_step=5.0,
    output_dt=1.0,
    face_average="harmonic",
    segment_at_data_knots=True,
)

sol = solve_q2(ENV_PATH, N=320, config=cfg)
print("Output shapes:", sol.T.shape, sol.C.shape)
print("Stored full-state shape:", sol.T_state.shape, sol.C_state.shape)
print("dr (cm):", sol.grid.dr * 100)
print("Solver stats:", sol.solver_stats)


## 5. Tables required in the paper

Rows correspond to 0.5–3.0 h and columns to 0, 0.5, 1.0, 1.5, 2.0 cm.


In [ ]:
T_table, C_table = paper_tables(sol)
display(T_table.round(4))
display(C_table.round(4))


## 6. Radial profiles

Use the stored 60 s full-state snapshots so the profiles are plotted on the actual FVM resolution instead of only the 0.1 cm result grid.


In [ ]:
profile_times = [1800, 3600, 5400, 7200, 9000, 10800]

plt.figure(figsize=(8, 4.5))
for t0 in profile_times:
    r_aug, u_aug = augmented_snapshot(sol, "T", t0)
    plt.plot(r_aug * 100, u_aug, label=f"{t0/3600:.1f} h")
plt.xlabel("Radius from center (cm)")
plt.ylabel("Temperature (degC)")
plt.title("Q2 radial temperature profiles")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4.5))
for t0 in profile_times:
    r_aug, u_aug = augmented_snapshot(sol, "C", t0)
    plt.plot(r_aug * 100, u_aug, label=f"{t0/3600:.1f} h")
plt.xlabel("Radius from center (cm)")
plt.ylabel("Moisture content (kg/kg)")
plt.title("Q2 radial moisture profiles")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 7. Center / middle / surface histories

Small late-time temperature reversals are not automatically numerical errors because the measured ambient temperature is not perfectly monotone near its plateau. Interpret them together with the ambient curve.


In [ ]:
idx_center = np.where(np.isclose(sol.r, 0.0))[0][0]
idx_mid = np.where(np.isclose(sol.r, 0.01))[0][0]
idx_surface = np.where(np.isclose(sol.r, 0.02))[0][0]

plt.figure(figsize=(8, 4.5))
plt.plot(sol.t / 3600, sol.T[:, idx_center], label="Center")
plt.plot(sol.t / 3600, sol.T[:, idx_mid], label="1 cm")
plt.plot(sol.t / 3600, sol.T[:, idx_surface], label="Surface")
plt.plot(sol.t / 3600, sol.env.T_inf(sol.t), "--", label="Ambient")
plt.xlabel("Time (h)")
plt.ylabel("Temperature (degC)")
plt.title("Q2 temperature histories")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(sol.t / 3600, sol.C[:, idx_center], label="Center")
plt.plot(sol.t / 3600, sol.C[:, idx_mid], label="1 cm")
plt.plot(sol.t / 3600, sol.C[:, idx_surface], label="Surface")
plt.plot(sol.t / 3600, sol.env.C_inf(sol.t), "--", label="Ambient datum")
plt.xlabel("Time (h)")
plt.ylabel("Moisture content / datum (kg/kg)")
plt.title("Q2 moisture histories")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 8. Coupling diagnostics: property ranges and diffusivity histories

This section shows what the nonlinear coupling actually changes during the first 3 h.


In [ ]:
print(property_ranges(sol))

D_center = moisture_diffusivity(sol.C[:, idx_center], sol.T[:, idx_center])
D_mid = moisture_diffusivity(sol.C[:, idx_mid], sol.T[:, idx_mid])
D_surface = moisture_diffusivity(sol.C[:, idx_surface], sol.T[:, idx_surface])

plt.figure(figsize=(8, 4.5))
plt.semilogy(sol.t / 3600, D_center, label="Center")
plt.semilogy(sol.t / 3600, D_mid, label="1 cm")
plt.semilogy(sol.t / 3600, D_surface, label="Surface")
plt.xlabel("Time (h)")
plt.ylabel("Diffusivity D (m^2/s)")
plt.title("State-dependent moisture diffusivity")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 9. Physical checks and local surface Picard cost

The production solver must preserve positive moisture and physically reasonable flux directions. The local surface Picard iteration should converge in only a few scalar iterations per RHS call.


In [ ]:
print("Physical diagnostics:")
print(physical_diagnostics(sol))
print("Surface Picard mean iterations:", sol.solver_stats["surface_mean_iterations"])
print("Surface Picard max iterations:", sol.solver_stats["surface_max_iterations"])


## 10. Independent Robin residuals

Production uses a quadratic surface derivative. For validation this routine uses an independent cubic derivative based on the surface plus the last three cell centers. Because the moisture initial condition is incompatible with its Robin condition at exactly \(t=0\), the post-\(t=0\) residual summaries are the meaningful ones.


In [ ]:
bres = boundary_residuals(sol)
summary_keys = [
    "heat_max_abs_after_t0",
    "moisture_max_abs_after_t0",
    "heat_max_relative_after_t0",
    "moisture_max_relative_after_t0",
]
print({k: bres[k] for k in summary_keys})

plt.figure(figsize=(8, 4))
plt.semilogy(bres["time"][1:] / 3600, np.abs(bres["heat_residual"][1:]) + 1e-30)
plt.xlabel("Time (h)")
plt.ylabel("Absolute residual (W/m^2)")
plt.title("Independent heat Robin residual")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4))
plt.semilogy(bres["time"][1:] / 3600, np.abs(bres["moisture_residual"][1:]) + 1e-30)
plt.xlabel("Time (h)")
plt.ylabel("Absolute residual")
plt.title("Independent moisture Robin residual")
plt.grid(True, alpha=0.3)
plt.show()


## 11. Balance checks

For Q2, do **not** copy Q1's cumulative heat-energy check \(\Delta\int \rho c_p T\,dV=\int q\,dt\). Because \(\rho(C)c_p(C)\) varies while the adopted PDE is \(\rho c_p T_t=\nabla\cdot(k\nabla T)\), that cumulative identity is not the model's conservation law.

Instead check the semi-discrete heat balance:

\[
\sum_i \rho_i c_{p,i}V_i\dot T_i + A_R q_R \approx 0.
\]

For the effective moisture equation, the integral of \(C\) can still be compared with cumulative surface flux.


In [ ]:
bal = balance_diagnostics(sol)
print("Heat semi-discrete max relative residual:", bal["heat_max_relative"])
print("Moisture instantaneous max relative residual:", bal["moisture_instantaneous_max_relative"])
print("Moisture final integral-balance relative residual:", bal["moisture_integral_final_relative"])
print("Moisture global-normalized max residual:", bal["moisture_integral_global_normalized_max"])

plt.figure(figsize=(8, 4))
plt.semilogy(bal["time"][1:] / 3600, np.abs(bal["moisture_integral_residual"][1:]) + 1e-30)
plt.xlabel("Time (h)")
plt.ylabel("Integral balance residual")
plt.title("Moisture integral-balance residual")
plt.grid(True, alpha=0.3)
plt.show()


## 12. Grid convergence: \(N=320\to640\to1280\)

This is the main spatial-accuracy evidence for the paper. Comparison is performed directly on the required `1 s × 0.1 cm` output grid.

This cell is more expensive than the baseline solve.


In [ ]:
conv_df, conv_solutions = grid_convergence_study(
    ENV_PATH,
    N_values=(320, 640, 1280),
    config=cfg,
    t_end=10800,
)
display(conv_df)


## 13. Time-integrator sensitivity

Tighten the BDF tolerances and reduce `max_step`. If this difference is much smaller than the grid difference, the spatial discretization remains the dominant numerical error.


In [ ]:
cfg_tight = SolverConfig(
    method="BDF",
    rtol=2e-10,
    atol=2e-12,
    max_step=1.0,
    output_dt=1.0,
    face_average="harmonic",
    segment_at_data_knots=True,
)
sol_tight = solve_q2(ENV_PATH, N=320, config=cfg_tight)
compare_solutions(sol, sol_tight)


## 14. Face-coefficient sensitivity

The main scheme uses harmonic face coefficients. Since Appendix 3 properties are smooth, `midpoint_property` should approach the same solution as the mesh is refined. This is a useful discretization-sensitivity check rather than a second production model.


In [ ]:
face_df, face_solutions = face_average_sensitivity(
    ENV_PATH,
    N=320,
    modes=("harmonic", "midpoint_property", "arithmetic"),
    base_config=cfg,
    t_end=10800,
)
display(face_df)


## 15. Auxiliary reference: backward Euler + converged block Picard

This solver is deliberately independent from the production BDF time integrator. At every fixed time layer it alternates:

\[
C^{(m)}\to \rho,c_p,k\to T^{(m+1)}
\to D(C^{(m)},T^{(m+1)})\to C^{(m+1)}
\]

until both fields converge.

Use a modest grid (for example \(N=160\)) because this is an error-analysis tool, not the production path.


In [ ]:
pic_df, bdf_ref, pic_solutions = picard_error_study(
    ENV_PATH,
    N=160,
    dt_values=(60.0, 30.0, 15.0),
    bdf_config=SolverConfig(rtol=1e-9, atol=1e-11, max_step=5.0, output_dt=1.0),
    t_end=10800,
)
display(pic_df)

# A first-order backward-Euler trend should be visible as dt is halved.


## 16. Quantify lagged one-pass splitting error

The converged block-Picard result is compared with an intentionally simplified one-pass update at the same fixed `dt`. Their difference estimates the error introduced if one freezes the parameters and performs only one T→C sweep per time step.


In [ ]:
split_df, pic_converged, pic_onepass = splitting_error_study(
    ENV_PATH,
    N=160,
    dt=30.0,
    t_end=10800,
)
display(split_df)


## 17. Optional Q1–Q2 comparison over the first 1800 s

If the validated Q1 `result1_generated.xlsx` is copied to `data/`, this cell compares Q1 and Q2 on exactly the same output grid. Only the material-property model changes.


In [ ]:
Q1_RESULT = PROJECT / "data" / "result1_generated.xlsx"

if Q1_RESULT.exists():
    q1T = pd.read_excel(Q1_RESULT, sheet_name="温度")
    q1C = pd.read_excel(Q1_RESULT, sheet_name="水分浓度")
    q1_t = q1T.iloc[:, 0].to_numpy(dtype=float)
    q1_r_cm = np.array([float(x) for x in q1T.columns[1:]], dtype=float)
    q1_T = q1T.iloc[:, 1:].to_numpy(dtype=float)
    q1_C = q1C.iloc[:, 1:].to_numpy(dtype=float)

    mask_q2 = sol.t <= 1800
    q2_t = sol.t[mask_q2]
    q2_r_cm = sol.r * 100
    assert np.allclose(q1_t, q2_t)
    assert np.allclose(q1_r_cm, q2_r_cm)

    dT = sol.T[mask_q2] - q1_T
    dC = sol.C[mask_q2] - q1_C
    print("Max |Q2-Q1| temperature:", np.max(np.abs(dT)))
    print("Max |Q2-Q1| moisture:", np.max(np.abs(dC)))

    plt.figure(figsize=(8, 4))
    plt.plot(q2_t / 60, dT[:, -1], label="Surface")
    plt.plot(q2_t / 60, dT[:, 0], label="Center")
    plt.xlabel("Time (min)")
    plt.ylabel("Q2 - Q1 temperature (degC)")
    plt.title("Q2 versus Q1 temperature upgrade")
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(q2_t / 60, dC[:, -1], label="Surface")
    plt.plot(q2_t / 60, dC[:, 0], label="Center")
    plt.xlabel("Time (min)")
    plt.ylabel("Q2 - Q1 moisture (kg/kg)")
    plt.title("Q2 versus Q1 moisture upgrade")
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()
else:
    print("Optional Q1 result not found:", Q1_RESULT)


## 18. Final production solve \(N=1280\)

Run this only after the baseline, convergence, and diagnostics are satisfactory. On a typical scientific Python environment this is still a one-dimensional problem, but it is the most expensive standard cell in the notebook.


In [ ]:
cfg_prod = SolverConfig(
    method="BDF",
    rtol=1e-9,
    atol=1e-11,
    max_step=5.0,
    output_dt=1.0,
    face_average="harmonic",
    segment_at_data_knots=True,
)
sol_prod = solve_q2(ENV_PATH, N=1280, config=cfg_prod)
print(sol_prod.solver_stats)
display(paper_tables(sol_prod)[0].round(4))
display(paper_tables(sol_prod)[1].round(4))


## 19. Generate `result2.xlsx`

If the official Appendix 3 template is available, put it in `data/result2_template.xlsx`. Otherwise the writer creates the required two-sheet structure. Numerical diagnostics must always use unrounded internal arrays; rounding is only for the submission workbook.


In [ ]:
TEMPLATE_PATH = PROJECT / "data" / "result2_template.xlsx"
if not TEMPLATE_PATH.exists():
    TEMPLATE_PATH = None

# Choose sol_prod after the final N=1280 cell has been run.
FINAL_SOL = sol_prod
OUT_PATH = OUT_DIR / "result2.xlsx"
write_result2(FINAL_SOL, OUT_PATH, template_path=TEMPLATE_PATH, decimals=4)
print("Saved:", OUT_PATH)


## 20. Recommended paper evidence

Use the following items in the Q2 numerical section:

1. the joint FVM flux discretization and the local coupled \((T_s,C_s)\) Robin Picard formulas;
2. the `N=320,640,1280` convergence table;
3. one sentence on tightened-BDF sensitivity;
4. semi-discrete heat balance and moisture integral-balance residuals;
5. block-Picard `dt` convergence toward BDF, showing that BDF is not a black-box-only result;
6. one temperature radial-profile figure and one moisture radial-profile figure;
7. the Q1–Q2 first-1800-s comparison if space allows.

Do not call `1 s` the numerical time step. It is the required output interval.
